# Notebook 1: Exploratory Data Analysis
**Project:** Pharmaceutical Demand Forecasting System
**Company:** Psychotropics India Limited (PIL)
**Intern:** [Your Name] | Role: Data Science / ML Intern


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('tab10')

# Load both datasets
df_raw = pd.read_csv('../data/raw/synthetic_sales_data.csv', parse_dates=['date'])
df_agg = pd.read_csv('../data/raw/aggregated_monthly_sales.csv', parse_dates=['date'])

print('Raw dataset shape:', df_raw.shape)
print('Aggregated dataset shape:', df_agg.shape)
df_raw.head()

## 1. Dataset Overview

In [ ]:
print('=== DATA TYPES ===')
print(df_raw.dtypes)
print('\n=== MISSING VALUES ===')
print(df_raw.isnull().sum())
print('\n=== BASIC STATS ===')
df_raw[['units_sold','revenue_inr','stock_start','stock_end']].describe().round(2)

In [ ]:
# Product & category distribution
print('Products:', df_raw['product_name'].nunique())
print('Regions:', df_raw['region'].nunique())
print('Date range:', df_raw['date'].min().date(), 'to', df_raw['date'].max().date())
print('\nCategory breakdown:')
print(df_raw.groupby('category')['product_name'].nunique())

## 2. Total Monthly Sales Trend (All Products)

In [ ]:
monthly_total = df_agg.groupby('date')['total_units_sold'].sum().reset_index()

fig, ax = plt.subplots()
ax.plot(monthly_total['date'], monthly_total['total_units_sold'], color='steelblue', linewidth=2, marker='o', markersize=3)
ax.fill_between(monthly_total['date'], monthly_total['total_units_sold'], alpha=0.15, color='steelblue')
ax.set_title('Total Units Sold Across All Products (Jan 2022 – Dec 2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Units Sold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../reports/figures/01_total_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Product-wise Sales Over Time

In [ ]:
products = df_agg['product_name'].unique()
fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

for i, prod in enumerate(products):
    sub = df_agg[df_agg['product_name'] == prod]
    axes[i].plot(sub['date'], sub['total_units_sold'], linewidth=2, color=sns.color_palette('tab10')[i])
    axes[i].fill_between(sub['date'], sub['total_units_sold'], alpha=0.1, color=sns.color_palette('tab10')[i])
    axes[i].set_title(prod, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Product-wise Monthly Units Sold (2022–2024)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/02_product_wise_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Revenue by Product (3-Year Total)

In [ ]:
revenue_by_product = df_agg.groupby('product_name')['total_revenue_inr'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(revenue_by_product.index, revenue_by_product.values, color=sns.color_palette('tab10', len(revenue_by_product)))
ax.set_title('Total 3-Year Revenue by Product (INR)', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue (INR)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
for bar, val in zip(bars, revenue_by_product.values):
    ax.text(val + 500, bar.get_y() + bar.get_height()/2, f'₹{val:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/03_revenue_by_product.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Regional Sales Distribution

In [ ]:
region_sales = df_raw.groupby('region')['units_sold'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar
ax1.bar(region_sales.index, region_sales.values, color=sns.color_palette('Blues_d', len(region_sales)))
ax1.set_title('Total Units Sold by Region', fontweight='bold')
ax1.set_ylabel('Units Sold')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie
ax2.pie(region_sales.values, labels=region_sales.index, autopct='%1.1f%%', startangle=140,
        colors=sns.color_palette('Blues_d', len(region_sales)))
ax2.set_title('Regional Market Share', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/04_regional_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Seasonality Analysis — Monthly Average

In [ ]:
monthly_avg = df_agg.groupby(['month', 'month_name'])['total_units_sold'].mean().reset_index()
monthly_avg = monthly_avg.sort_values('month')

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly_avg['month_name'], monthly_avg['total_units_sold'], color='coral', edgecolor='white')
ax.axhline(monthly_avg['total_units_sold'].mean(), color='steelblue', linestyle='--', linewidth=1.5, label='Average')
ax.set_title('Average Monthly Demand (Seasonality Pattern)', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Units Sold')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/05_seasonality_monthly_avg.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Heatmap (Numeric Features)

In [ ]:
corr_cols = ['units_sold', 'revenue_inr', 'stock_start', 'stock_end', 'month', 'year', 'lag_1_units', 'lag_3_units', 'rolling_avg_3']
corr = df_raw[corr_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlation of features with units_sold:')
print(corr['units_sold'].drop('units_sold').sort_values(ascending=False))

## 8. Stockout Analysis

In [ ]:
stockout_by_product = df_raw.groupby('product_name')['stockout_flag'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(stockout_by_product.index, stockout_by_product.values, color='tomato', edgecolor='white')
ax.set_title('Stockout Events per Product (2022–2024)', fontsize=14, fontweight='bold')
ax.set_ylabel('No. of Stockout Months (across regions)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../reports/figures/07_stockout_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

total_stockout_rate = df_raw['stockout_flag'].mean() * 100
print(f'Overall stockout rate: {total_stockout_rate:.2f}%')

## 9. Year-over-Year Growth

In [ ]:
yoy = df_agg.groupby(['year', 'product_name'])['total_units_sold'].sum().reset_index()
yoy_pivot = yoy.pivot(index='product_name', columns='year', values='total_units_sold')
yoy_pivot['YoY_2022_23'] = ((yoy_pivot[2023] - yoy_pivot[2022]) / yoy_pivot[2022] * 100).round(1)
yoy_pivot['YoY_2023_24'] = ((yoy_pivot[2024] - yoy_pivot[2023]) / yoy_pivot[2023] * 100).round(1)
print('Year-over-Year Growth (%)')
print(yoy_pivot[['YoY_2022_23', 'YoY_2023_24']].to_string())

## 10. Save Cleaned Data

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)

# Drop rows where lag features are NaN (first 3 months per group) for ML use
df_clean = df_raw.dropna(subset=['lag_1_units', 'lag_3_units', 'rolling_avg_3']).copy()
df_clean.to_csv('../data/processed/cleaned_sales_data.csv', index=False)

df_agg_clean = df_agg.dropna(subset=['lag_1_total', 'lag_3_total', 'rolling_avg_3']).copy()
df_agg_clean.to_csv('../data/processed/cleaned_aggregated_data.csv', index=False)

print(f'Cleaned raw data saved: {df_clean.shape}')
print(f'Cleaned aggregated data saved: {df_agg_clean.shape}')
print('EDA complete. All figures saved to reports/figures/')